# Test complet — Qwen3.6 27B FP8 + finegrained-fp8 V4

Objectif : faire **Run All** et obtenir un diagnostic complet sans tests manuels cellule par cellule.

Le notebook :
1. vérifie Python / Torch / CUDA / Transformers / Kernels ;
2. localise automatiquement `finegrained-fp8/v4` ;
3. force `LOCAL_KERNELS` vers la V4 locale ;
4. vérifie que `kernels` résout bien la V4 sans passer par le Hub ;
5. localise le modèle Qwen3.6 ;
6. inspecte `config.json`, la quantification et les classes AutoModel ;
7. charge le processor ;
8. charge le modèle FP8 ;
9. teste une vraie inférence multimodale sur une page PDF si un PDF est trouvé ;
10. affiche le texte brut, les tokens générés et un rapport final.

**Important :** redémarrer le workspace/kernel avant `Run All` afin d'avoir une VRAM propre.


## 1 — Configuration et diagnostic environnement

In [ ]:
import os, sys, json, glob, gc, time, traceback, importlib
from pathlib import Path

# Evite la fragmentation CUDA avant import torch
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPORT = {}
ERRORS = []

def ok(k, v=True):
    REPORT[k] = v
    print(f"✅ {k}: {v}")

def ko(k, e):
    REPORT[k] = False
    ERRORS.append((k, repr(e)))
    print(f"❌ {k}: {type(e).__name__}: {e}")

print("="*100)
print("ENVIRONNEMENT")
print("="*100)
print("Python :", sys.version)

for pkg in ["torch","transformers","kernels","accelerate","safetensors","PIL","fitz","psutil"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:15} :", getattr(m, "__version__", "OK"))
    except Exception as e:
        print(f"{pkg:15} : ABSENT -> {e}")


## 2 — Torch / CUDA / GPU / VRAM

In [ ]:
import torch

ok("torch import", torch.__version__)
print("Torch CUDA build :", torch.version.cuda)
print("CUDA disponible  :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA indisponible : impossible de tester Qwen FP8.")

print("GPU              :", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

free_b, total_b = torch.cuda.mem_get_info()
print(f"VRAM libre       : {free_b/1024**3:.2f} GB")
print(f"VRAM totale      : {total_b/1024**3:.2f} GB")

REPORT["GPU"] = torch.cuda.get_device_name(0)
REPORT["VRAM_libre_GB_avant"] = round(free_b/1024**3,2)
REPORT["VRAM_totale_GB"] = round(total_b/1024**3,2)


## 3 — Détection et verrouillage de finegrained-fp8 V4

In [ ]:
from pathlib import Path
import kernels
import kernels.utils

REPO_ID = "kernels-community/finegrained-fp8"

# Chemin connu dans ton Domino + recherche de secours
known = Path("/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community/finegrained-fp8/v4")
candidates = [known]
candidates += [Path(x) for x in glob.glob("/domino/**/finegrained-fp8/v4", recursive=True)]

V4_PATH = next((p for p in candidates if p.exists() and (p/"build"/"torch-cuda").exists()), None)
if V4_PATH is None:
    raise FileNotFoundError("Impossible de trouver finegrained-fp8/v4 avec build/torch-cuda.")

V4_PATH = V4_PATH.resolve()
os.environ["LOCAL_KERNELS"] = f"{REPO_ID}={V4_PATH}"

# Le parser est caché par lru_cache : on vide son cache après changement de variable
if hasattr(kernels.utils._parse_local_kernel_overrides, "cache_clear"):
    kernels.utils._parse_local_kernel_overrides.cache_clear()

print("kernels version :", getattr(kernels, "__version__", "inconnue"))
print("V4              :", V4_PATH)
print("LOCAL_KERNELS   :", os.environ["LOCAL_KERNELS"])

overrides = kernels.utils._get_local_kernel_overrides()
print("Overrides       :", overrides)

assert REPO_ID in overrides, "Override finegrained-fp8 absent."
assert Path(overrides[REPO_ID]).resolve() == V4_PATH, "Override ne pointe pas vers V4."
assert (V4_PATH/"build"/"torch-cuda").exists()

ok("LOCAL_KERNELS -> V4", str(V4_PATH))


## 4 — Chargement direct du kernel local V4

In [ ]:
from kernels import get_kernel

try:
    # Avec LOCAL_KERNELS, get_kernel retourne get_local_kernel(override)
    # avant le contrôle trust_remote_code et avant tout téléchargement Hub.
    fp8_kernel = get_kernel(REPO_ID, version=4, backend="cuda")
    kernel_file = str(getattr(fp8_kernel, "__file__", ""))
    print("Module :", fp8_kernel)
    print("File   :", kernel_file)

    if "/v4/" not in kernel_file.replace("\\","/"):
        raise RuntimeError(f"Kernel chargé mais origine /v4/ non confirmée : {kernel_file}")

    ok("finegrained-fp8 V4 chargé", kernel_file)
except Exception as e:
    ko("finegrained-fp8 V4 chargé", e)
    raise


## 5 — Détection automatique du modèle Qwen3.6 27B FP8

In [ ]:
# Recherche large, sans dépendre d'un nom de dossier exact.
roots = [
    "/domino/edv/modelhub",
    "/mnt/data",
]

model_candidates = []
for root in roots:
    if not Path(root).exists():
        continue
    for cfg in glob.glob(root + "/**/config.json", recursive=True):
        p = Path(cfg).parent
        name = str(p).lower()
        if "qwen" in name and ("3.6" in name or "3_6" in name or "qwen3" in name) and ("27" in name or "27b" in name):
            model_candidates.append(p)

# Fallback : tout Qwen 27B dont config contient une quantification FP8.
if not model_candidates:
    for root in roots:
        if not Path(root).exists():
            continue
        for cfg in glob.glob(root + "/**/config.json", recursive=True):
            p = Path(cfg).parent
            name = str(p).lower()
            if "qwen" in name and "27" in name:
                try:
                    c = json.load(open(cfg, encoding="utf-8"))
                    if "fp8" in json.dumps(c).lower():
                        model_candidates.append(p)
                except:
                    pass

# Déduplique
model_candidates = list(dict.fromkeys(map(lambda x: x.resolve(), model_candidates)))

print("Candidats trouvés :")
for p in model_candidates:
    print(" -", p)

if not model_candidates:
    raise FileNotFoundError(
        "Aucun Qwen3.x/3.6 27B FP8 trouvé automatiquement sous /domino/edv/modelhub."
    )

# Préférence aux chemins contenant fp8 puis v4/main
model_candidates.sort(
    key=lambda p: (
        "fp8" not in str(p).lower(),
        "v4" not in str(p).lower(),
        "main" not in str(p).lower(),
        len(str(p))
    )
)

MODEL_PATH = str(model_candidates[0])
print("\nMODEL_PATH retenu :", MODEL_PATH)
ok("MODEL_PATH", MODEL_PATH)


## 6 — Inspection config.json et architecture réelle

In [ ]:
cfg_path = Path(MODEL_PATH)/"config.json"
cfg = json.load(open(cfg_path, encoding="utf-8"))

print(json.dumps({
    "model_type": cfg.get("model_type"),
    "architectures": cfg.get("architectures"),
    "auto_map": cfg.get("auto_map"),
    "quantization_config": cfg.get("quantization_config"),
    "torch_dtype": cfg.get("torch_dtype"),
}, indent=2, ensure_ascii=False))

REPORT["model_type"] = cfg.get("model_type")
REPORT["architectures"] = cfg.get("architectures")
REPORT["quantization_config"] = cfg.get("quantization_config")

if "fp8" not in json.dumps(cfg.get("quantization_config", {})).lower():
    print("⚠️ La quantization_config ne contient pas littéralement 'fp8'. On continue pour inspecter le modèle réel.")
else:
    ok("config quantification FP8", True)


## 7 — Détection des AutoModel disponibles

In [ ]:
import transformers
from transformers import AutoConfig, AutoProcessor

print("Transformers :", transformers.__version__)

auto_names = [
    "AutoModelForMultimodalLM",
    "AutoModelForImageTextToText",
    "AutoModelForVision2Seq",
    "AutoModelForCausalLM",
]

available = {}
for name in auto_names:
    cls = getattr(transformers, name, None)
    available[name] = cls
    print(f"{name:30} :", "DISPONIBLE" if cls else "ABSENT")

config = AutoConfig.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True,
)

print("\nConfig class :", config.__class__.__name__)
print("model_type   :", getattr(config, "model_type", None))

ok("AutoConfig", config.__class__.__name__)


## 8 — Chargement du processor

In [ ]:
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True,
)
print("Processor class :", processor.__class__.__name__)
print(f"Temps           : {time.time()-t0:.1f}s")
ok("AutoProcessor", processor.__class__.__name__)


## 9 — Choix de la classe modèle + chargement FP8

In [ ]:
# Priorité adaptée aux Transformers récents.
# On ne force PAS AutoModelForVision2Seq si la config du Qwen3.6 ne lui appartient pas.
priority = [
    "AutoModelForMultimodalLM",
    "AutoModelForImageTextToText",
    "AutoModelForVision2Seq",
]

ModelClass = None
chosen_name = None

for name in priority:
    cls = available.get(name)
    if cls is None:
        continue
    try:
        # Test de mapping sans charger 27B : from_config peut toutefois allouer un modèle,
        # donc on inspecte d'abord le mapping interne quand disponible.
        mapping = getattr(cls, "_model_mapping", None)
        if mapping is None or type(config) in mapping.keys():
            ModelClass = cls
            chosen_name = name
            break
    except Exception:
        continue

# Pour un modèle remote-code, AutoModelForMultimodalLM reste le choix attendu dans cet environnement.
if ModelClass is None and available.get("AutoModelForMultimodalLM") is not None:
    ModelClass = available["AutoModelForMultimodalLM"]
    chosen_name = "AutoModelForMultimodalLM"

if ModelClass is None:
    raise RuntimeError("Aucune classe AutoModel multimodale compatible détectée.")

print("Classe retenue :", chosen_name)
REPORT["AutoModel"] = chosen_name

gc.collect()
torch.cuda.empty_cache()

free_b, total_b = torch.cuda.mem_get_info()
print(f"VRAM libre juste avant modèle : {free_b/1024**3:.2f} GB")

t0 = time.time()

try:
    model = ModelClass.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
        trust_remote_code=True,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
except ValueError as e:
    # Certains remote-code nécessitent explicitement trust du kernel.
    # LOCAL_KERNELS doit normalement court-circuiter cette étape.
    print("Premier chargement ValueError :", e)
    raise

model.eval()
print("Model class   :", model.__class__.__name__)
print(f"Chargé en     : {time.time()-t0:.1f}s")

free_b, total_b = torch.cuda.mem_get_info()
allocated = torch.cuda.memory_allocated(0)/1024**3
reserved = torch.cuda.memory_reserved(0)/1024**3

print(f"VRAM libre    : {free_b/1024**3:.2f} GB")
print(f"VRAM allouée  : {allocated:.2f} GB")
print(f"VRAM réservée : {reserved:.2f} GB")

REPORT["model_class"] = model.__class__.__name__
REPORT["VRAM_allouee_GB"] = round(allocated,2)
ok("Chargement modèle", model.__class__.__name__)


## 10 — Recherche d'un PDF de test et rendu première page

In [ ]:
import fitz
from PIL import Image

pdf_candidates = []
for root in ["/mnt/data", "/domino/datasets", "/domino/projects"]:
    if Path(root).exists():
        pdf_candidates.extend(glob.glob(root + "/**/*.pdf", recursive=True))

print("PDF trouvés :", len(pdf_candidates))

TEST_PDF = pdf_candidates[0] if pdf_candidates else None

if TEST_PDF:
    print("PDF utilisé :", TEST_PDF)
    doc = fitz.open(TEST_PDF)
    page = doc[0]
    pix = page.get_pixmap(matrix=fitz.Matrix(1.5,1.5), alpha=False)
    image = Image.frombytes("RGB", [pix.width,pix.height], pix.samples)
    print("Image :", image.size)
    REPORT["PDF_test"] = TEST_PDF
else:
    image = None
    print("⚠️ Aucun PDF trouvé automatiquement. Le modèle est chargé ; le test image sera ignoré.")


## 11 — Inférence multimodale réelle (si PDF trouvé)

In [ ]:
RAW_OUTPUT = None

if image is not None:
    prompt = """Lis cette page de document bancaire avec précision.
Retourne uniquement un JSON valide avec cette structure :
{
  "type_document": null,
  "texte_visible": null,
  "noms": [],
  "dates": [],
  "montants": [],
  "references": []
}
N'invente aucune valeur. Si une donnée est absente, utilise null ou [].
"""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    # Qwen multimodal récent : chat template du processor.
    if hasattr(processor, "apply_chat_template"):
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = prompt

    # Essai API processor moderne, puis fallback Qwen classique.
    try:
        inputs = processor(
            text=[text],
            images=[image],
            return_tensors="pt",
            padding=True,
        )
    except Exception:
        inputs = processor(
            text=text,
            images=image,
            return_tensors="pt",
        )

    # Déplacement uniquement des tenseurs ; respecte les types d'origine.
    target_device = next(model.parameters()).device
    inputs = {
        k: (v.to(target_device) if hasattr(v, "to") else v)
        for k,v in inputs.items()
    }

    print("Inputs :", {k: tuple(v.shape) if hasattr(v,"shape") else type(v).__name__ for k,v in inputs.items()})

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
        )

    # Décoder uniquement les nouveaux tokens quand input_ids existe.
    if "input_ids" in inputs and generated.ndim == 2:
        prompt_len = inputs["input_ids"].shape[1]
        generated_only = generated[:, prompt_len:]
    else:
        generated_only = generated

    RAW_OUTPUT = processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    print("\n" + "="*100)
    print("SORTIE BRUTE DU MODELE")
    print("="*100)
    print(RAW_OUTPUT)

    REPORT["sortie_longueur"] = len(RAW_OUTPUT)
    REPORT["sortie_preview"] = RAW_OUTPUT[:1000]

    # Détection simple de dégénérescence
    low = RAW_OUTPUT.lower()
    bad_patterns = ["/sec/sec/sec", "qual qual qual"]
    degenerate = any(x in low for x in bad_patterns)
    REPORT["sortie_degeneree"] = degenerate

    if degenerate:
        print("\n❌ SORTIE DEGENEREE détectée.")
    else:
        print("\n✅ Pas de motif de dégénérescence connu détecté.")
else:
    print("Test d'inférence ignoré : aucun PDF disponible.")


## 12 — Validation JSON de la sortie

In [ ]:
def extract_json(text):
    if not text:
        return None
    text = text.strip()
    text = text.replace("```json","").replace("```","").strip()
    try:
        return json.loads(text)
    except:
        pass
    a, b = text.find("{"), text.rfind("}")
    if a >= 0 and b > a:
        try:
            return json.loads(text[a:b+1])
        except:
            return None
    return None

if RAW_OUTPUT is not None:
    parsed = extract_json(RAW_OUTPUT)
    print("JSON valide :", parsed is not None)
    if parsed is not None:
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
        ok("JSON valide", True)
    else:
        REPORT["JSON valide"] = False
        print("❌ Le modèle produit du texte, mais pas un JSON valide.")
else:
    print("Pas de sortie à parser.")


## 13 — Rapport final unique

In [ ]:
print("\n" + "="*100)
print("RAPPORT FINAL")
print("="*100)

for k,v in REPORT.items():
    print(f"{k:32} : {v}")

if ERRORS:
    print("\nERREURS :")
    for k,e in ERRORS:
        print(" -", k, ":", e)

print("\n" + "="*100)

critical = [
    REPORT.get("LOCAL_KERNELS -> V4") not in (None, False),
    REPORT.get("finegrained-fp8 V4 chargé") not in (None, False),
    REPORT.get("Chargement modèle") not in (None, False),
]

if all(critical):
    print("✅ KERNEL V4 + MODELE : CHARGEMENT OK")
    if "sortie_degeneree" in REPORT:
        if REPORT["sortie_degeneree"]:
            print("❌ INFERENCE : sortie dégénérée — problème situé après le chargement du modèle.")
        else:
            print("✅ INFERENCE : sortie non dégénérée.")
else:
    print("❌ Un composant critique n'a pas été validé.")

print("="*100)
